In [ ]:
# OrchAMP — TCGA-BRCA Survival Prediction  (LATE FUSION / EB-PCA)
# One cluster per modality — each modality denoised independently.
# Train/test split loaded from frozen CSVs generated by
#   generate_brca_survival_splits.ipynb  — run that first.
import sys
from pathlib import Path
from datetime import datetime
import json

import numpy as np
import pandas as pd

In [ ]:
sys.path.append("../Python_scripts")

import importlib
amp_mod = importlib.import_module("multimodal_prediction_survival")
amp_mod = importlib.reload(amp_mod)

MultimodalClusterAllUPipeline       = amp_mod.MultimodalClusterAllUPipeline
train_linear_cox_survival           = amp_mod.train_linear_cox_survival
predict_survival_from_test_data_all = amp_mod.predict_survival_from_test_data_all

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
matrices_dir = Path("../matrices")
data_dir     = Path("../data")
splits_dir   = Path("../splits")
results_dir  = Path("../results/brca_survival/late_fusion")
results_dir.mkdir(parents=True, exist_ok=True)

SPLIT_TAG = "brca_survival"

K_RNA   = 10
K_METH  = 8
K_CNV   = 5

SIMILARITY   = "cka"
NUM_CLUSTERS = 3      # late fusion: one cluster per modality (RNA, Meth, CNV)
AMP_ITERS    = 10
RANDOM_STATE = 42

COX_EPOCHS  = 1000
COX_MC      = 16
COX_LR      = 1e-3
PRINT_EVERY = 100

print("Results dir:", results_dir.resolve())

In [ ]:
# ── Load frozen split ────────────────────────────────────────────────────────
sid_path = splits_dir / f"{SPLIT_TAG}_sample_ids.csv"
trn_path = splits_dir / f"{SPLIT_TAG}_train_idx.csv"
tst_path = splits_dir / f"{SPLIT_TAG}_test_idx.csv"

for p in [sid_path, trn_path, tst_path]:
    if not p.exists():
        raise FileNotFoundError(f"{p} not found — run generate_brca_survival_splits.ipynb first.")

sample_ids = pd.read_csv(sid_path)["sample_id"].tolist()
idx_tr     = pd.read_csv(trn_path)["index"].values
idx_te     = pd.read_csv(tst_path)["index"].values

print(f"Loaded {len(sample_ids)} samples  |  train {len(idx_tr)}  |  test {len(idx_te)}")

In [ ]:
# ── Load matrices in frozen sample order ─────────────────────────────────────
print("Loading matrices...", flush=True)
rna_df   = pd.read_csv(matrices_dir / "RNA_X_full.csv",          index_col=0)
meth_df  = pd.read_csv(matrices_dir / "Methylation_X_full.csv",  index_col=0)
# CNV: preprocessed LD features — run preprocess_cnv_brca.ipynb first
cnv_df   = pd.read_csv(matrices_dir / "CNV_X_ld_features.csv",  index_col=0)
surv_raw = pd.read_csv(data_dir / "BRCA_survival.tsv", sep="\t", index_col=0)
surv_df  = surv_raw[["OS", "OS.time"]].copy()
surv_df.columns = ["event", "time"]
surv_df["event"] = pd.to_numeric(surv_df["event"], errors="coerce")
surv_df["time"]  = pd.to_numeric(surv_df["time"],  errors="coerce")
surv_df = surv_df.dropna(subset=["event", "time"])
surv_df = surv_df[surv_df["time"] > 0]

X_rna  = rna_df.loc[sample_ids].values.astype(np.float32)
X_meth = meth_df.loc[sample_ids].values.astype(np.float32)
X_cnv  = cnv_df.loc[sample_ids].values.astype(np.float32)
event  = surv_df.loc[sample_ids, "event"].values.astype(np.float32)
time   = surv_df.loc[sample_ids, "time"].values.astype(np.float32)

for name, arr in [("RNA", X_rna), ("Meth", X_meth), ("CNV", X_cnv)]:
    assert not np.any(np.isnan(arr)), f"NaN in {name}"
assert not np.any(np.isnan(event)), "NaN in event — re-run generate_brca_survival_splits.ipynb"
assert not np.any(np.isnan(time)),  "NaN in time  — re-run generate_brca_survival_splits.ipynb"
assert np.all(time > 0),            "Non-positive survival time found"

print(f"RNA {X_rna.shape}  Meth {X_meth.shape}  CNV {X_cnv.shape}")
print(f"Events: {int(event.sum())} / {len(event)}")

In [ ]:
# ── Slice train / test ───────────────────────────────────────────────────────
X_rna_tr,  X_rna_te  = X_rna[idx_tr],  X_rna[idx_te]
X_meth_tr, X_meth_te = X_meth[idx_tr], X_meth[idx_te]
X_cnv_tr,  X_cnv_te  = X_cnv[idx_tr],  X_cnv[idx_te]
event_tr,  event_te   = event[idx_tr],  event[idx_te]
time_tr,   time_te    = time[idx_tr],   time[idx_te]

print(f"Train: {len(idx_tr)}  ({int(event_tr.sum())} events)")
print(f"Test : {len(idx_te)}  ({int(event_te.sum())} events)")

In [ ]:
# ── Arrange modalities ───────────────────────────────────────────────────────
B_hd_train = [X_rna_tr, X_meth_tr]
B_hd_test  = [X_rna_te, X_meth_te]
A_ld_train = [X_cnv_tr]
A_ld_test  = [X_cnv_te]
K_hd_list  = [K_RNA, K_METH]

print("HD ranks:", K_hd_list, "  LD rank (CNV):", K_CNV)

In [ ]:
# ── Fit AMP pipeline ─────────────────────────────────────────────────────────
pipe = MultimodalClusterAllUPipeline()
pipe.task         = "survival"
pipe.y_surv_train = (event_tr, time_tr)

print("[AMP] Fitting PCA (HD)...", flush=True)
pipe.fit_pca_highdim(B_hd_train, K_hd_list, preprocess=False)

print("[AMP] Fitting LD loadings (CNV)...", flush=True)
pipe.fit_lowdim(A_ld_train, top_features=None)

print("[AMP] Clustering (late fusion: one cluster per modality)...", flush=True)
labels = pipe.cluster_all_modalities(
    X_list_hd=B_hd_train, method=SIMILARITY,
    num_clusters=NUM_CLUSTERS, threshold=None
)
print(f"Cluster labels: {labels}  →  {len(set(labels))} cluster(s)")

print("[AMP] Building EB cluster models...", flush=True)
pipe.build_cluster_models(B_hd_train)

print("[AMP] Running AMP...", flush=True)
amp_res = pipe.run_amp(X_list_hd=B_hd_train, amp_iters=AMP_ITERS)

In [ ]:
# ── Train Cox head ───────────────────────────────────────────────────────────
cox_models, log_risk_train = train_linear_cox_survival(
    pipe, X_hd=B_hd_train, X_ld=A_ld_train,
    event=event_tr, time=time_tr,
    epochs=COX_EPOCHS, lr=COX_LR, mc_samples=COX_MC,
    seed=RANDOM_STATE, print_every=PRINT_EVERY,
)
print("Train log-risk: min={:.4f}  max={:.4f}".format(log_risk_train.min(), log_risk_train.max()))

In [ ]:
# ── Predict on TEST ──────────────────────────────────────────────────────────
_, _, log_risk_test = predict_survival_from_test_data_all(
    pipe, X_test_hd=B_hd_test, X_test_ld=A_ld_test, mc_samples=50, seed=RANDOM_STATE
)
print("Test log-risk: min={:.4f}  max={:.4f}".format(log_risk_test.min(), log_risk_test.max()))

In [ ]:
# ── Evaluate ─────────────────────────────────────────────────────────────────
from lifelines.utils import concordance_index

c_test  = concordance_index(time_te, -log_risk_test,  event_te)
c_train = concordance_index(time_tr, -log_risk_train, event_tr)

print(f"C-index  TRAIN: {c_train:.4f}")
print(f"C-index  TEST : {c_test:.4f}")

In [ ]:
# ── Save ─────────────────────────────────────────────────────────────────────
np.save(results_dir / "log_risk_test.npy",  log_risk_test)
np.save(results_dir / "log_risk_train.npy", log_risk_train)
np.save(results_dir / "event_test.npy",     event_te)
np.save(results_dir / "time_test.npy",      time_te)
np.save(results_dir / "idx_test.npy",       idx_te.astype(int))

pd.DataFrame({
    "sample_id": np.array(sample_ids)[idx_te],
    "log_risk":  log_risk_test,
    "event":     event_te,
    "time":      time_te,
}).to_csv(results_dir / "test_predictions.csv", index=False)

stamp = datetime.now().isoformat(timespec="seconds").replace(":", "-")
metrics = dict(
    fusion="late", num_clusters=NUM_CLUSTERS,
    c_index_test=float(c_test), c_index_train=float(c_train),
    n_train=int(len(idx_tr)), n_test=int(len(idx_te)),
    n_events_test=int(event_te.sum()), K_RNA=K_RNA, K_METH=K_METH, K_CNV=K_CNV,
    cox_epochs=COX_EPOCHS, cox_mc=COX_MC, timestamp=stamp,
)
with open(results_dir / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved to:", results_dir.resolve())
print(json.dumps(metrics, indent=2))

In [ ]:
# ── Kaplan-Meier (median risk split) ─────────────────────────────────────────
try:
    from lifelines import KaplanMeierFitter
    import matplotlib.pyplot as plt

    med = np.median(log_risk_test)
    hi, lo = log_risk_test >= med, log_risk_test < med

    fig, ax = plt.subplots(figsize=(6, 4))
    for mask, label, color in [(hi, "High risk", "C1"), (lo, "Low risk", "C0")]:
        KaplanMeierFitter().fit(
            time_te[mask], event_observed=event_te[mask], label=label
        ).plot_survival_function(ax=ax, ci_show=True, color=color)

    ax.set_title(f"EB-PCA Late Fusion — TCGA-BRCA  (C-index={c_test:.3f})")
    ax.set_xlabel("Days"); ax.set_ylabel("Survival probability")
    plt.tight_layout()
    fig.savefig(results_dir / "km_plot.pdf", bbox_inches="tight")
    plt.show()
except ImportError:
    print("lifelines not installed — skipping KM plot.")